# 查看模型名字

In [ ]:
from openai import OpenAI
client = OpenAI(base_url="http://192.168.31.33:1234/v1", api_key="lm-studio")
print(client.models.list())


参数设置参考文档：https://lmstudio.ai/docs/developer/openai-compat/chat-completions

## LM Studio / Ollama

In [ ]:
from openai import OpenAI

# # 1) 连接到 LM Studio 的 OpenAI 兼容接口
# client = OpenAI(base_url="http://26.26.26.1:1234/v1", api_key="lm-studio")  # 这里随便填，LM Studio 一般不校验
# # 2) 填你在 LM Studio /v1/models 里看到的模型名（必须匹配）
# MODEL = "qwen/qwen3-14b"  # 你也可以改成 LM Studio 显示的实际 model id


# 1) 连接到 Ollama 的 OpenAI 兼容接口
client = OpenAI(base_url="http://wanglei_ollama:11434/v1", api_key="ollama")  # 这里随便填，LM Studio 一般不校验
# 2) 填你在 ollama复制的模型名（必须匹配）
MODEL = "qwen3:14b"

# 3) 直接在代码里写好对话内容
messages = [ {"role": "system", "content": "你是一个有帮助的助手。"},
             {"role": "user", "content": "你是什么型号的模型？有多少B的参数？4B还是8B /no_think"},]

# 4) 只调用一次
resp = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    # temperature=0.9,
)

print("Assistant:", resp.choices[0].message.content)


# 循环读取csv数据 / 格式化输出

## 读取csv文件

In [ ]:
import pandas as pd

# 方式1：CSV 和 notebook 在同一目录
df = pd.read_csv("../data/3.数据合并/裁判文书网2000-2021_1_4.csv")

# 查看条数
print("数据数量：", len(df))

# 查看前5行
df.head()

In [ ]:
# 第0行
row_Select = df.iloc[200]

row_Select

In [ ]:
# 第x行
row_Select = df.iloc[32]

# 提取字段
case_number = row_Select["案号"]
court_name = row_Select["法院"]
court_area = row_Select["所属地区"]
case_type1 = row_Select["案件类型"]
case_type2 = row_Select["案由"]
judgment_date = row_Select["裁判日期"]
privy = row_Select["当事人"]
judgment = row_Select["全文"]

result = f"法院：{court_name}。所属地区：{court_area}。全文：{judgment}。"

print(result,"\n")
# 截断字数
result = (result or "")[:1000]
print(result)


根据案号中的省份缩写，法院名称，所属地区，以及全文信息推断：案件发生的时间、省份、城市、行政区、具体地点

## 将读入csv作为prompt

In [ ]:
import os
import json
from openai import OpenAI

client = OpenAI(base_url="https://api.zetatechs.com/v1", api_key=os.environ["LLM_API_KEY"])
MODEL = "gemini-2.0-flash-001"

messages = [{"role": "system",
             "content": ("""你是专业的法律文件分析助理。请从提供的文本中提取交通事故发生的时间、省份、城市、行政区、具体地点、交通工具、是否死亡。
                            请从提供的文本中提取信息，并严格返回以下JSON格式。如果有字段无法准确提取，请返回空字段。请勿添加任何其他格式或标记：\n"
    {"case_time" : "YYYY年MM月DD日HH时(案件发生时间,不是判决落款时间、不是书记员时间、不是立案时间)",
     "province" : "案件发生省份",
     "city" : "案件发生城市",
     "district" : "案件发生行政区",
     "specific_place" : "案件发生具体地点（尽量用原文说法若无法精确到街道请返回null）",
     "vehicle" : "0=汽车事故；1=汽车与摩托/电动车；2=汽车与行人；3=摩托/电动车与行人",
     "death" : "0=无人死亡；1=有人死亡"}""")},
    {"role": "user",
     "content": (result)}]

resp = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=0,
    response_format={"type": "json_object"})

raw = resp.choices[0].message.content
data = json.loads(raw)
print(json.dumps(data, ensure_ascii=False, indent=2))


# 合并代码

In [ ]:
import os
import json
from openai import OpenAI
import pandas as pd

# 读取数据
df = pd.read_csv("../data/3.数据合并/裁判文书网2000-2021_last1w.csv")

# 第x行
row_Select = df.iloc[33]

# 提取字段
case_number = row_Select["案号"]
court_name = row_Select["法院"]
court_area = row_Select["所属地区"]
case_type1 = row_Select["案件类型"]
case_type2 = row_Select["案由"]
judgment_date = row_Select["裁判日期"]
privy = row_Select["当事人"]
judgment = row_Select["全文"]

result = f"法院：{court_name}。所属地区：{court_area}。全文：{judgment}。"
result = (result or "")[:1000]  # 截断文字前1000


client = OpenAI(base_url="https://api.zetatechs.com/v1", api_key=os.environ["LLM_API_KEY"])
MODEL = "gemini-2.0-flash-001"

messages = [{"role": "system",
             "content": ("""你是专业的法律文件分析助理。请从提供的文本中提取交通事故发生的时间、省份、城市、行政区、具体地点、交通工具、是否死亡。
                            请从提供的文本中提取信息，并严格返回以下JSON格式。如果有字段无法准确提取，请返回空字段。请勿添加任何其他格式或标记：\n"
    {"case_time" : "YYYY年MM月DD日HH时(案件发生时间,不是判决落款时间、不是书记员时间、不是立案时间)",
     "province" : "案件发生省份",
     "city" : "案件发生城市",
     "district" : "案件发生行政区",
     "specific_place" : "案件发生具体地点（尽量用原文说法若无法精确到街道请返回null）",
     "vehicle" : "0=汽车事故；1=汽车与摩托/电动车；2=汽车与行人；3=摩托/电动车与行人",
     "death" : "0=无人死亡；1=有人死亡"}""")},
    {"role": "user",
     "content": (result)}]

resp = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=0,
    response_format={"type": "json_object"})

raw = resp.choices[0].message.content
data = json.loads(raw)[0]
print(data)



# 循环处理

In [ ]:
import os
import json
from openai import OpenAI
import pandas as pd

# ---------- 路径 ----------
in_path = "../data/3.数据合并/裁判文书网2000-2021_last1w.csv"
out_path = "../data/4.数据处理/交通事故结构化结果.csv"

# ---------- 读取 ----------
df = pd.read_csv(in_path)

# ---------- OpenAI Client ----------
client = OpenAI(
    base_url="https://api.zetatechs.com/v1",
    api_key=os.environ["LLM_API_KEY"]
)
MODEL = "gemini-2.0-flash-001"

SYSTEM_PROMPT = """你是专业的法律文件分析助理。请从提供的文本中提取交通事故发生的时间、省份、城市、行政区、具体地点、交通工具、是否死亡。
请从提供的文本中提取信息，并严格返回以下JSON格式。如果有字段无法准确提取，请返回空字段。请勿添加任何其他格式或标记：
{"case_time": "YYYY年MM月DD日HH时(案件发生时间,不是判决落款时间、不是书记员时间、不是立案时间)",
  "province": "案件发生省份",
  "city": "案件发生城市",
  "district": "案件发生行政区",
  "specific_place": "案件发生具体地点（尽量用原文说法若无法精确到街道请返回空字段）",
  "vehicle": "0=汽车事故；1=汽车与摩托/电动车；2=汽车与行人；3=摩托/电动车与行人",
  "death": "0=无人死亡；1=有人死亡"}
"""

def call_llm(row):
    court_name = row.get("法院")
    court_area = row.get("所属地区")
    judgment = row.get("全文")

    text = f"法院：{court_name}。所属地区：{court_area}。全文：{judgment}。"
    text = (text or "")[:1000] # 截断文字前1000

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": text}],
        temperature=0,
        response_format={"type": "json_object"})

    raw = resp.choices[0].message.content
    obj = json.loads(raw)

    # 兼容：有些模型可能返回 [ {...} ]
    if isinstance(obj, list):
        obj = obj[0] if obj else {}

    return obj

for i, row in df.iterrows():
    info = call_llm(row)

    # 可选：保留原始定位字段，便于回溯
    record = {"row_id": i, "案号": row.get("案号"), "裁判日期": row.get("裁判日期"), "当事人": row.get("当事人"),**info}

    file_exists = os.path.exists(out_path)
    pd.DataFrame([record]).to_csv(
        out_path,
        mode="a",
        index=False,
        header=not file_exists,
        encoding="utf-8-sig")

    print(f"saved: {i+1}/{len(df)}")
